<a href="https://colab.research.google.com/github/Adriandzidan/Tugas-Pak-Prayit-PBO/blob/main/jobsheet11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Jika dijalankan di Google Colab, install dependensi dulu
!pip -q install pandas streamlit

In [ ]:
import os
import sqlite3
import datetime
import locale
import pandas as pd

try:
    locale.setlocale(locale.LC_ALL, 'id_ID.UTF-8')
except locale.Error:
    try:
        locale.setlocale(locale.LC_ALL, 'Indonesian_Indonesia.1252')
    except:
        print('Locale Indonesia tidak tersedia.')

def format_rp(angka):
    try:
        return locale.currency(angka or 0, grouping=True, symbol='Rp ')[:-3]
    except:
        return f'Rp {angka or 0:,.0f}'.replace(',', '.')

Locale Indonesia tidak tersedia.


In [ ]:
BASE_DIR = os.getcwd()
NAMA_DB = 'pengeluaran_harian.db'
DB_PATH = os.path.join(BASE_DIR, NAMA_DB)

KATEGORI_PENGELUARAN = [
    'Makanan',
    'Transportasi',
    'Hiburan',
    'Tagihan',
    'Belanja',
    'Kesehatan',
    'Pendidikan',
    'Lainnya'
]

KATEGORI_DEFAULT = 'Lainnya'
print('DB path:', DB_PATH)

DB path: /content/pengeluaran_harian.db


In [ ]:
class Transaksi:
    """Merepresentasikan satu transaksi pengeluaran."""

    def __init__(self, deskripsi: str, jumlah: float, kategori: str, tanggal: datetime.date | str, id_transaksi: int | None = None):
        self.id = id_transaksi
        self.deskripsi = str(deskripsi) if deskripsi else 'Tanpa Deskripsi'

        try:
            jumlah_float = float(jumlah)
            self.jumlah = jumlah_float if jumlah_float > 0 else 0.0
        except (ValueError, TypeError):
            self.jumlah = 0.0

        self.kategori = str(kategori) if kategori else 'Lainnya'

        if isinstance(tanggal, datetime.date):
            self.tanggal = tanggal
        elif isinstance(tanggal, str):
            try:
                self.tanggal = datetime.datetime.strptime(tanggal, '%Y-%m-%d').date()
            except ValueError:
                self.tanggal = datetime.date.today()
        else:
            self.tanggal = datetime.date.today()

    def __repr__(self):
        return (
            f'Transaksi(ID:{self.id}, '
            f'Tanggal:{self.tanggal}, '
            f'Jumlah:{self.jumlah}, '
            f'Kategori:{self.kategori}, '
            f'Deskripsi:{self.deskripsi})'
        )

    def to_dict(self):
        return {
            'deskripsi': self.deskripsi,
            'jumlah': self.jumlah,
            'kategori': self.kategori,
            'tanggal': self.tanggal.strftime('%Y-%m-%d')
        }

In [ ]:
def get_db_connection():
    """Membuka koneksi ke database SQLite."""
    try:
        conn = sqlite3.connect(DB_PATH, timeout=10)
        conn.row_factory = sqlite3.Row
        return conn
    except sqlite3.Error as e:
        print(f'ERROR koneksi database: {e}')
        return None

def execute_query(query: str, params: tuple = None):
    """Menjalankan query INSERT, UPDATE, DELETE."""
    conn = get_db_connection()
    if not conn:
        return None

    try:
        cursor = conn.cursor()

        if params:
            cursor.execute(query, params)
        else:
            cursor.execute(query)

        conn.commit()
        return cursor.lastrowid if cursor.lastrowid != 0 else True

    except sqlite3.Error as e:
        print(f'ERROR query gagal: {e}')
        conn.rollback()
        return None

    finally:
        conn.close()

def fetch_query(query: str, params: tuple = None, fetch_all: bool = True):
    """Menjalankan query SELECT."""
    conn = get_db_connection()
    if not conn:
        return None

    try:
        cursor = conn.cursor()

        if params:
            cursor.execute(query, params)
        else:
            cursor.execute(query)

        return cursor.fetchall() if fetch_all else cursor.fetchone()

    except sqlite3.Error as e:
        print(f'ERROR fetch gagal: {e}')
        return None

    finally:
        conn.close()

def get_dataframe(query: str, params: tuple = None) -> pd.DataFrame:
    """Mengambil data SELECT menjadi DataFrame Pandas."""
    conn = get_db_connection()
    if not conn:
        return pd.DataFrame()

    try:
        return pd.read_sql_query(query, conn, params=params)

    except Exception as e:
        print(f'ERROR dataframe gagal: {e}')
        return pd.DataFrame()

    finally:
        conn.close()

def setup_database_initial():
    """Membuat tabel transaksi jika belum ada."""
    conn = get_db_connection()
    if not conn:
        return False

    try:
        cursor = conn.cursor()

        sql_create_table = """
        CREATE TABLE IF NOT EXISTS transaksi (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            deskripsi TEXT NOT NULL,
            jumlah REAL NOT NULL CHECK(jumlah > 0),
            kategori TEXT,
            tanggal DATE NOT NULL
        );
        """

        cursor.execute(sql_create_table)
        conn.commit()
        print('Tabel transaksi siap.')
        return True

    except sqlite3.Error as e:
        print(f'ERROR setup database: {e}')
        return False

    finally:
        conn.close()

setup_database_initial()

Tabel transaksi siap.


True

In [ ]:
class AnggaranHarian:
    """Mengelola logika bisnis pengeluaran harian."""

    _db_setup_done = False

    def __init__(self):
        if not AnggaranHarian._db_setup_done:
            print('[AnggaranHarian] Mengecek database...')

            if setup_database_initial():
                AnggaranHarian._db_setup_done = True
                print('[AnggaranHarian] Database siap.')
            else:
                print('[AnggaranHarian] Database gagal dibuat.')

    def tambah_transaksi(self, transaksi: Transaksi) -> bool:
        if not isinstance(transaksi, Transaksi) or transaksi.jumlah <= 0:
            return False

        sql = """
        INSERT INTO transaksi (deskripsi, jumlah, kategori, tanggal)
        VALUES (?, ?, ?, ?)
        """

        params = (
            transaksi.deskripsi,
            transaksi.jumlah,
            transaksi.kategori,
            transaksi.tanggal.strftime('%Y-%m-%d')
        )

        last_id = execute_query(sql, params)

        if last_id is not None:
            transaksi.id = last_id
            return True

        return False

    def hapus_transaksi(self, id_transaksi: int) -> bool:
        try:
            id_transaksi = int(id_transaksi)
        except (TypeError, ValueError):
            return False

        if id_transaksi < 1:
            return False

        sql = 'DELETE FROM transaksi WHERE id = ?'
        result = execute_query(sql, (id_transaksi,))
        return result is not None

    def get_semua_transaksi_obj(self) -> list[Transaksi]:
        sql = """
        SELECT id, deskripsi, jumlah, kategori, tanggal
        FROM transaksi
        ORDER BY tanggal DESC, id DESC
        """

        rows = fetch_query(sql, fetch_all=True)
        transaksi_list = []

        if rows:
            for row in rows:
                transaksi_list.append(
                    Transaksi(
                        id_transaksi=row['id'],
                        deskripsi=row['deskripsi'],
                        jumlah=row['jumlah'],
                        kategori=row['kategori'],
                        tanggal=row['tanggal']
                    )
                )

        return transaksi_list

    def get_dataframe_transaksi(self, filter_tanggal: datetime.date | None = None) -> pd.DataFrame:
        query = """
        SELECT id, tanggal, kategori, deskripsi, jumlah
        FROM transaksi
        """

        params = None

        if filter_tanggal:
            query += ' WHERE tanggal = ?'
            params = (filter_tanggal.strftime('%Y-%m-%d'),)

        query += ' ORDER BY tanggal DESC, id DESC'

        df = get_dataframe(query, params=params)

        if not df.empty:
            df['Jumlah (Rp)'] = df['jumlah'].map(lambda x: f'Rp {x or 0:,.0f}'.replace(',', '.'))
            df = df[['id', 'tanggal', 'kategori', 'deskripsi', 'Jumlah (Rp)']]

        return df

    def hitung_total_pengeluaran(self, tanggal: datetime.date | None = None) -> float:
        sql = 'SELECT SUM(jumlah) FROM transaksi'
        params = None

        if tanggal:
            sql += ' WHERE tanggal = ?'
            params = (tanggal.strftime('%Y-%m-%d'),)

        result = fetch_query(sql, params=params, fetch_all=False)

        if result and result[0] is not None:
            return float(result[0])

        return 0.0

    def get_pengeluaran_per_kategori(self, tanggal: datetime.date | None = None) -> dict:
        hasil = {}

        sql = """
        SELECT kategori, SUM(jumlah)
        FROM transaksi
        """

        params = []

        if tanggal:
            sql += ' WHERE tanggal = ?'
            params.append(tanggal.strftime('%Y-%m-%d'))

        sql += """
        GROUP BY kategori
        HAVING SUM(jumlah) > 0
        ORDER BY SUM(jumlah) DESC4/0AdkVLPwA4bAkCZ4TsYc1OVo9YmU5XMiuYbwX7FG2DxQUA8onexE4nGXQWhDe61tLBZNVnw
        """

        rows = fetch_query(sql, params=tuple(params) if params else None, fetch_all=True)

        if rows:
            for row in rows:
                kategori = row['kategori'] if row['kategori'] else 'Lainnya'
                jumlah = float(row[1]) if row[1] is not None else 0.0
                hasil[kategori] = jumlah

        return hasil

anggaran = AnggaranHarian()

[AnggaranHarian] Mengecek database...
Tabel transaksi siap.
[AnggaranHarian] Database siap.


In [ ]:
# Contoh pemakaian di Colab
tx = Transaksi('Makan siang', 25000, 'Makanan', datetime.date.today())
print('Tambah:', anggaran.tambah_transaksi(tx))
print('ID baru:', tx.id)

df = anggaran.get_dataframe_transaksi()
df

Tambah: True
ID baru: 2


,id,tanggal,kategori,deskripsi,Jumlah (Rp)
0,2,2026-06-11,Makanan,Makan siang,Rp 25.000
1,1,2026-06-11,Makanan,Makan siang,Rp 25.000


In [ ]:
# Contoh hapus transaksi
# Ganti angka di bawah dengan ID yang benar jika ingin menghapus data
# print(anggaran.hapus_transaksi(1))